# Projeto 01: Crédito Inteligente

<h3>Integrantes:</h3> Leonardo Nakashima, João Pedro Paulino, Isadora Jardim, Jefferson Lopes e Maria Eduarda

## Importando as bibliotecas 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
from datetime import datetime
import openpyxl
import os

# Criando o dataFrame com o credito_banco.csv

In [ ]:
df = pd.read_csv('credito_banco.csv')

## ETAPA 1: Limpeza dos dados com pandas

In [ ]:
print("=== ETAPA 1: Limpeza dos dados com pandas ===\n")

print(f"Dataset original: {df.shape}")
print(f"Valores ausentes:\n{df.isnull().sum()}")

# Removend valores nulos
df_clean = df.dropna()
print(f"\nApós remover valores ausentes: {df_clean.shape}")

# Removendo rendas outliers
Q1 = df_clean['renda_anual'].quantile(0.25)
Q3 = df_clean['renda_anual'].quantile(0.75)
IQR = Q3 - Q1
limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean['renda_anual'] < limite_inf) | (df_clean['renda_anual'] > limite_sup)]
print(f"Outliers de renda detectados: {len(outliers)}")

df_clean = df_clean[(df_clean['renda_anual'] >= limite_inf) & (df_clean['renda_anual'] <= limite_sup)]
print(f"Dataset após remover outliers: {df_clean.shape}")

print(f"\nDistribuição de inadimplência:")
print(f"  Adimplentes: {(df_clean['inadimplente'] == 0).sum()}")
print(f"  Inadimplentes: {(df_clean['inadimplente'] == 1).sum()}")
print(f"  Taxa: {df_clean['inadimplente'].mean()*100:.1f}%")


#  ETAPA 2: Gerando gráficos
## Gerando histograma com Matplotlib

In [ ]:
adimplentes = df_clean[df_clean['inadimplente'] == 0]['renda_anual']
inadimplentes = df_clean[df_clean['inadimplente'] == 1]['renda_anual']

plt.figure()

plt.hist (adimplentes, bins=30, alpha=0.5, label='Adimplentes', color='#3D003B')
plt.hist (inadimplentes, bins=30, alpha=0.5, label='Inadimplentes', color='#D1F07F')
plt.xlabel('Renda Anual')
plt.ylabel('Frequência')
plt.title('Distribuição de Renda: Adimplentes vs Inadimplentes')
plt.legend()

plt.show()

## Gerando heatmap com Seaborn

In [ ]:
cols = ['renda_anual', 'idade', 'divida_atual', 'score_externo']
corr = df_clean[cols].corr()

plt.figure()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)

plt.title('Mapa de Correlação')
plt.show()

## ETAPA 3: Treinar Robô e Automatizar a exportação.

In [ ]:
print("\n=== ETAPA 3.1: Regras para Criação Realista ===")

def classificar_risco(row):
    pontuacao_risco = 0
    
    #Quanto maior a renda, menor a chance de inadimplência
    if row['renda_anual'] > 150000:
        pontuacao_risco -= 1
    elif row['renda_anual'] < 50000:
        pontuacao_risco += 1
        
    #Quanto maior o histórico de pagamento, menor a chance
    if row['historico_pagamento'] > 70:
        pontuacao_risco -= 1
    elif row['historico_pagamento'] < 30:
        pontuacao_risco += 1

    #Quanto maior a dívida atual, maior a chance   
    if row['divida_atual'] > 20000:
        pontuacao_risco += 1
    elif row['divida_atual'] < 5000:
        pontuacao_risco -= 1
        
    # pontuação > 1 prevemos inadimplência (1)
    return 1 if pontuacao_risco >= 1 else 0

# aplica as regras solicitado na base de dados (item 3.1 da atividade)
y_pred = df_clean.apply(classificar_risco, axis=1)
y_test = df_clean['inadimplente']

#calcula o Recall e o F1-score da classe 1
acuracia = accuracy_score(y_test, y_pred)
recall_inadimplente = recall_score(y_test, y_pred)
f1_inadimplente = f1_score(y_test, y_pred)

print(f"Acurácia Geral das Regras: {acuracia:.2%}")
print(f"Recall (Classe 1 - Inadimplentes): {recall_inadimplente:.2%}")
print(f"F1-Score (Classe 1 - Inadimplentes): {f1_inadimplente:.2%}")

print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred))

#-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-#
print("\n=== ETAPA 3.2: Automatização RPA (Exportação) ===")

# aplica as regras na base de dados limpa para exportação
df_clean['previsao_inadimplencia'] = y_pred

lista_alto_risco = df_clean[df_clean['previsao_inadimplencia'] == 1]

data_atual = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
nome_arquivo = f"relatorio_risco_{data_atual}.xlsx" #pega a data pra ter nomes sempre diferentes

try:
    lista_alto_risco.to_excel(nome_arquivo, index=False)
    print(f"Sucesso! Relatório '{nome_arquivo}' gerado com {len(lista_alto_risco)} clientes de alto risco.")
    print(f"Caminho do arquivo: {os.path.abspath(nome_arquivo)}")
except Exception as e:
    print(f"Erro ao exportar:  {e}")
